# The User-Facing LLM Engine API

In our last notebook, we assembled the `EngineCore`, the powerful heart of our inference system that orchestrates scheduling and model execution. However, the `EngineCore` is a low-level component designed for batch processing; it's like a raw, high-performance car engine. To actually drive the car, you need a steering wheel, pedals, and a dashboard.

This is where the `LLMEngine` comes in. It provides the user-friendly "dashboard" for our system, allowing users to submit prompts easily and receive generated text back in a smooth, continuous stream. By the end of this notebook, you will understand how to build this high-level API, creating an asynchronous, streaming interface that makes interacting with the powerful `EngineCore` simple and intuitive.

In [ ]:
import time
import queue
import threading
import uuid
from collections import deque
from dataclasses import dataclass, field

# --- Mock Implementations from Previous Notebooks ---
# These simplified classes represent the components we've built so far.

@dataclass
class MiniRequest:
    """A simplified request, containing prompt tokens and tracking generated tokens."""
    request_id: str
    prompt_tokens: list[int]
    max_new_tokens: int = 20
    tokens_generated: list[int] = field(default_factory=list)
    finished: bool = False

    def __repr__(self):
        return f"Request(id={self.request_id}, generated={len(self.tokens_generated)}/{self.max_new_tokens})"

class MiniScheduler:
    """A scheduler that decides which requests run next."""
    def __init__(self):
        self.request_queue = deque()
        self.running_requests = {}

    def add_request(self, request: MiniRequest):
        self.request_queue.append(request)
        print(f"Scheduler: Queued {request.request_id}")

    def schedule(self) -> list[MiniRequest]:
        # Simple logic: run up to 2 requests at a time.
        # Move queued requests to running.
        while self.request_queue and len(self.running_requests) < 2:
            req = self.request_queue.popleft()
            self.running_requests[req.request_id] = req
            print(f"Scheduler: Moved {req.request_id} to running.")

        # Return the batch of requests to execute.
        return list(self.running_requests.values())

    def on_request_finished(self, request_id: str):
        if request_id in self.running_requests:
            del self.running_requests[request_id]
            print(f"Scheduler: Completed {request_id}.")

class MiniExecutor:
    """A mock executor that "runs" the model."""
    def __init__(self):
        # A fake vocabulary for demonstration
        self.vocabulary = [" a", " cat", " sat", " on", " mat", " the", " dog", " chased", " squirrel", "."]
        self.token_map = {word: i for i, word in enumerate(self.vocabulary)}
        self.next_token_idx = 0

    def tokenize(self, text: str) -> list[int]:
        # A very basic tokenizer
        return [self.token_map.get(word, 0) for word in text.split()]

    def detokenize(self, tokens: list[int]) -> str:
        return "".join([self.vocabulary[t] for t in tokens])

    def execute_batch(self, batch: list[MiniRequest]) -> dict[str, int]:
        """Simulates a forward pass, generating one new token for each request."""
        time.sleep(0.1) # Simulate the time it takes to run the model
        output_tokens = {}
        for req in batch:
            # Generate a new token in a deterministic, cyclical way for fun
            new_token_id = (sum(req.prompt_tokens) + len(req.tokens_generated)) % len(self.vocabulary)
            output_tokens[req.request_id] = new_token_id
        return output_tokens

@dataclass
class EngineCoreOutput:
    request_id: str
    new_token_id: int
    finished: bool = False

class MiniEngineCore:
    """The orchestration layer from the previous notebook."""
    def __init__(self):
        self.scheduler = MiniScheduler()
        self.executor = MiniExecutor()

    def add_request(self, request_id: str, prompt_tokens: list[int], max_new_tokens: int):
        req = MiniRequest(
            request_id=request_id,
            prompt_tokens=prompt_tokens,
            max_new_tokens=max_new_tokens
        )
        self.scheduler.add_request(req)

    def step(self) -> list[EngineCoreOutput]:
        scheduled_batch = self.scheduler.schedule()
        if not scheduled_batch:
            return []

        # Execute the model on the batch
        new_tokens_map = self.executor.execute_batch(scheduled_batch)

        # Process the results
        outputs = []
        for req in scheduled_batch:
            if req.request_id not in new_tokens_map:
                continue

            new_token = new_tokens_map[req.request_id]
            req.tokens_generated.append(new_token)

            # Check for completion
            if len(req.tokens_generated) >= req.max_new_tokens:
                req.finished = True
                self.scheduler.on_request_finished(req.request_id)

            outputs.append(EngineCoreOutput(req.request_id, new_token, req.finished))

        return outputs

    def tokenize(self, text):
        return self.executor.tokenize(text)

    def detokenize(self, tokens):
        return self.executor.detokenize(tokens)

### The Core Idea: The Restaurant Maître d'

Imagine our `EngineCore` is a world-class, but highly specialized, kitchen. The chefs (the `Executor`) are incredibly fast, and the lead chef (the `Scheduler`) knows exactly how to organize orders to keep the kitchen running at peak efficiency. However, they only understand "kitchen-speak"—complex orders for batched items. You can't just walk into the kitchen and say "I'd like a steak."

The `LLMEngine` is the maître d' of this restaurant.

1.  **Takes Your Order:** You, the customer, simply say, "I'd like a prompt about a cat." The maître d' (`LLMEngine`) takes this simple request.
2.  **Translates for the Kitchen:** The maître d' translates your request into a formal kitchen ticket (`MiniRequest`) that the `Scheduler` can understand. It assigns it a unique order number (`request_id`).
3.  **Manages Communication:** You don't want to wait 20 minutes and get your entire meal at once. You'd prefer appetizers, then the main course. The maître d' manages this. As soon as a part of your order is ready (a new token is generated), they bring it to your table. This is **streaming**.
4.  **Hides the Complexity:** You never see the kitchen's chaotic, batched, and optimized process. You just have a pleasant conversation with the maître d', and food appears at your table as it's ready.

The `LLMEngine` provides this seamless, streaming experience by running the `EngineCore` in the background and using a communication system (like message queues) to deliver results to the user as soon as they are generated.

In [ ]:
class UserLLMEngine:
    """The user-facing API for the LLM engine."""

    def __init__(self, engine_core: MiniEngineCore):
        self.engine_core = engine_core
        self.request_outputs = {}  # Maps request_id to a Queue
        self._is_running = True

        # The "engine room" worker thread
        self.stepper_thread = threading.Thread(target=self._engine_stepper_thread)
        self.stepper_thread.start()

    def _engine_stepper_thread(self):
        """Continuously steps the engine core in the background."""
        while self._is_running:
            outputs = self.engine_core.step()
            for out in outputs:
                if out.request_id in self.request_outputs:
                    self.request_outputs[out.request_id].put(out)
            # Prevent busy-waiting when there are no requests
            if not outputs and not self.engine_core.scheduler.request_queue:
                time.sleep(0.01)

    def generate(self, prompt: str, max_new_tokens: int = 10):
        """Submits a prompt and yields generated tokens as they arrive."""
        request_id = f"req-{uuid.uuid4().hex[:8]}"
        prompt_tokens = self.engine_core.tokenize(prompt)
        output_queue = queue.Queue()
        self.request_outputs[request_id] = output_queue

        self.engine_core.add_request(request_id, prompt_tokens, max_new_tokens)

        # Yield tokens as they are put into the queue
        finished = False
        while not finished:
            output = output_queue.get() # This will block until an item is available
            new_token_str = self.engine_core.detokenize([output.new_token_id])
            yield new_token_str
            finished = output.finished

        del self.request_outputs[request_id]

    def shutdown(self):
        self._is_running = False
        self.stepper_thread.join()
        print("\nEngine has been shut down.")

Let's walk through that implementation.

-   **`__init__(self, engine_core)`**: The engine is initialized with an instance of our `MiniEngineCore`. It creates a dictionary `request_outputs`, which will act as our "mailbox" system. Each request gets its own `queue.Queue` in this dictionary to receive its results. Crucially, it starts a background thread, `stepper_thread`, which immediately begins running the `_engine_stepper_thread` function.

-   **`_engine_stepper_thread(self)`**: This is the heart of the asynchronous behavior. It's a simple loop that runs for the entire lifetime of the engine.
    1.  It calls `engine_core.step()`, which, as we know, runs one batch of inference.
    2.  It iterates through the outputs from that step.
    3.  For each output, it looks up the corresponding request's "mailbox" (its queue) in `self.request_outputs`.
    4.  It `put`s the output object into the queue. This is like a mail carrier delivering a letter to the correct mailbox.
    This thread's only job is to turn the crank of the engine and deliver the mail. It's completely decoupled from the user's code.

-   **`generate(self, prompt, ...)`**: This is the beautiful, simple public API. This is what the user calls.
    1.  It generates a unique `request_id`.
    2.  It creates a new `queue.Queue` for this specific request and stores it in the `request_outputs` mailbox dictionary.
    3.  It calls `engine_core.add_request()`, formally submitting the request to the scheduler.
    4.  Now, the magic: it enters a `while` loop and calls `output_queue.get()`. This call **blocks**. The user's code pauses right here, waiting patiently for a "letter" to arrive in its mailbox.
    5.  As soon as the background `_engine_stepper_thread` puts an output into the queue, this `get()` call unblocks, receives the new token, and the loop continues.
    6.  It uses `yield` to return the new token string to the user, making this method a generator. The user can simply loop over it.
    7.  Once an output with `finished=True` is received, the loop terminates and the request is cleaned up from the mailbox dictionary.

This design cleanly separates the user's interaction (calling `generate` and iterating) from the engine's continuous, batch-processing operation.

In [ ]:
# --- Demonstration ---

# 1. Set up the core engine
core = MiniEngineCore()

# 2. Wrap it in our user-friendly API
engine = UserLLMEngine(engine_core=core)

# 3. Submit requests and stream the results
print("--- Submitting two requests concurrently ---")

# The `generate` method returns a generator, so we need to iterate over it
# to actually see the output.
generator1 = engine.generate("a cat sat", max_new_tokens=5)
generator2 = engine.generate("the dog chased", max_new_tokens=7)

print("\nRequest 1 output: ", end="")
for token in generator1:
    print(token, end="", flush=True)
    time.sleep(0.05) # Add a small delay to make streaming visible

print("\nRequest 2 output: ", end="")
for token in generator2:
    print(token, end="", flush=True)
    time.sleep(0.05) # Add a small delay to make streaming visible

# 4. Clean up the engine
engine.shutdown()

### Going Deeper: Input and Output Processors

In our simplified `UserLLMEngine`, the `generate` method handles everything: creating the request object, tokenizing the prompt, detokenizing the output, and managing the streaming loop. The real vLLM architecture is more modular and introduces two key helper classes: `InputProcessor` and `OutputProcessor`.

-   **`InputProcessor`**: Its sole responsibility is to convert user-provided inputs into the `EngineCoreRequest` format that the `EngineCore` understands. This is like the restaurant's host who takes your verbal order and writes it neatly on a standardized order slip. This layer would handle:
    -   Tokenizing a raw string prompt.
    -   Applying a chat template if the input is a list of messages (e.g., `[{"role": "user", "content": "Hi!"}]`).
    -   Validating sampling parameters (`temperature`, `top_p`, etc.).
    -   Handling multi-modal inputs (e.g., processing an image).

-   **`OutputProcessor`**: Its job is the reverse. It takes the raw `EngineCoreOutput` (which just contains token IDs) and transforms it back into a user-friendly format. This is like the waiter who plates the food beautifully before serving. This layer handles:
    -   Detokenizing the token IDs into human-readable text.
    -   Checking for stop strings (e.g., stop generating if you see the word "Human:").
    -   Managing the state of all unfinished requests, tracking which tokens belong to which request over time.
    -   Formatting the final output object that the user receives.

Why this separation? It's a powerful design principle called **Separation of Concerns**. The `LLMEngine` becomes a pure orchestrator, delegating the specialized tasks of input "translation" and output "formatting" to dedicated components. This makes the whole system cleaner, easier to test, and more extensible. The `EngineCore` doesn't need to know what a "chat template" is, and the `InputProcessor` doesn't need to know how the KV cache works.

In [ ]:
# --- Visualization of the Data Flow ---

def visualize_engine_flow():
    """ASCII art visualization of the LLMEngine data flow."""
    print("User Code".center(30) + "UserLLMEngine".center(40) + "Background Thread".center(30))
    print("-" * 100)
    print("for token in generate('...'):".ljust(30))
    print(" " * 30 + "1. generate() creates request_id & queue".ljust(40))
    print(" " * 30 + "|".ljust(40))
    print(" " * 30 + "V".ljust(40))
    print(" " * 30 + "2. core.add_request(...)".ljust(40))
    print(" " * 30 + "|".ljust(40) + "while True:".center(30))
    print(" " * 30 + "V".ljust(40) + "    outputs = core.step()".center(30))
    print(" " * 30 + "EngineCore".center(40) + "    for out in outputs:".center(30))
    print(" " * 30 + "(Scheduler + Executor)".center(40) + "       queue.put(out) ---+ ".center(30))
    print(" " * 30 + " ".ljust(40) + " " * 30)
    print(" " * 30 + " ".ljust(40) + " " * 30)
    print("    [BLOCKS WAITING]".ljust(30))
    print(" " * 30 + "4. queue.get() <----------+" .ljust(40))
    print(" " * 30 + "|".ljust(40))
    print(" " * 30 + "V".ljust(40))
    print("    print(token)".ljust(30) + "5. yield detokenize(out)".ljust(40))
    print("    ^".ljust(30))
    print("    |".ljust(30))
    print("    +-------------------------+".ljust(30))

visualize_engine_flow()

### Summary

In this final notebook, we built the friendly face of our LLM serving system: the `UserLLMEngine`. We successfully bridged the gap between a complex, batch-oriented internal engine and the simple, streaming experience that a user expects.

**What we built:**
-   A high-level `UserLLMEngine` class that wraps the `MiniEngineCore`.
-   An asynchronous, non-blocking architecture using a background thread to drive the core engine.
-   A streaming `generate` method that uses a queue to deliver tokens to the user as soon as they are created.

**Key Takeaways:**
-   **Abstraction is Key:** The `LLMEngine` provides a crucial abstraction layer, hiding the intricate details of scheduling and execution from the end-user.
-   **Threads and Queues for Streaming:** The combination of a dedicated worker thread and a message queue per request is a classic and powerful pattern for building responsive, streaming APIs. The main thread submits a job and waits on a queue, while the worker thread does the heavy lifting and delivers results.
-   **Separation of Concerns:** In real-world systems, functionality is broken down even further into components like `InputProcessor` and `OutputProcessor`, leading to more robust and maintainable code.

This completes our journey through the core mental model of vLLM. We've seen how requests are scheduled, how memory is managed with PagedAttention (conceptually), how the `EngineCore` orchestrates the process, and finally, how the `LLMEngine` presents it all to the user in a clean, elegant package.